# CRAG · Corrective RAG

> Notebook generated from [`examples/rag/01_crag.py`](../../examples/rag/01_crag.py).

| Data | Value |
|------|-------|
| **Dataset** | SQuAD 2.0 (embedded — Wikipedia contexts) |
| **API key** | ⚠️  **Requires API key** (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`) in `.env`. |

**Expected result:** 5-step pipeline per query: retrieve, grade, filter, decide, generate. Prints `CRAGResult.answer`, sources, and `relevance_scores`.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
CRAG — Corrective RAG with LLM-based relevance grading
=========================================================
Architecture: SPEC-005 / prismal.rag.crag

Dataset: SQuAD 2.0 (Stanford Question Answering Dataset)
  • 150,000 question-answer pairs about Wikipedia articles.
  • Reference: https://huggingface.co/datasets/rajpurkar/squad_v2
  • Why: CRAG requires base documents + questions about them.
    SQuAD 2.0 provides exactly this: Wikipedia contexts and
    real user questions with extracted answers.

CRAG architecture description:
  5-step pipeline:
  1. RETRIEVE   — ChromaDB similarity_search(query, k=5)
  2. GRADE      — LLM scores each chunk 0.0-1.0 by relevance
  3. FILTER     — Keeps chunks with relevance_score >= 0.5
  4. DECIDE     — If all are filtered out → trigger web fallback (stub)
  5. GENERATE   — LLM answers with context + source citations

Correction flow:
  - Irrelevant chunks are discarded before generation (avoids hallucinations).
  - If no chunks are relevant, the web fallback prevents empty answers.
  - The LLM cites the sources used in the final answer.

Usage:
    uv run python examples/rag/01_crag.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio

from langchain_core.documents import Document

from prismal.rag.crag import CRAGPipeline, CRAGResult
from prismal.rag.vector_store import ChromaVectorStore

## Dataset: SQuAD 2.0 fragments (Wikipedia contexts)

In [ ]:
# Real Wikipedia contexts used in SQuAD 2.0.
SQUAD_CONTEXTS = [
    {
        "source": "wikipedia_transformers",
        "title": "Transformer (machine learning model)",
        "content": (
            "The transformer is a deep learning architecture that relies on the parallel "
            "multi-head attention mechanism. Introduced by Vaswani et al. in 2017 in the "
            "paper 'Attention Is All You Need', the transformer architecture has become "
            "the standard for natural language processing tasks. Unlike recurrent neural "
            "networks (RNNs), transformers process entire sequences simultaneously, "
            "enabling highly parallelizable training. The key innovation is the "
            "self-attention mechanism that allows the model to weigh the importance "
            "of different words in the input sequence when generating each output token."
        ),
    },
    {
        "source": "wikipedia_bert",
        "title": "BERT (language model)",
        "content": (
            "BERT (Bidirectional Encoder Representations from Transformers) is a "
            "language model developed by Google in 2018. BERT uses a transformer "
            "architecture and is pre-trained on masked language modeling and "
            "next sentence prediction tasks using the BooksCorpus and English "
            "Wikipedia datasets. BERT was revolutionary because it achieved "
            "state-of-the-art results on 11 NLP tasks. Unlike GPT which is "
            "unidirectional, BERT processes text bidirectionally, allowing it "
            "to understand context from both left and right of each token."
        ),
    },
    {
        "source": "wikipedia_python",
        "title": "Python (programming language)",
        "content": (
            "Python is a high-level, general-purpose programming language. Its design "
            "philosophy emphasizes code readability using indentation. Python is "
            "dynamically typed and garbage-collected. It supports multiple programming "
            "paradigms, including structured, object-oriented, and functional programming. "
            "Guido van Rossum began working on Python in the late 1980s as a successor "
            "to the ABC programming language. Python 3.0 was released in 2008. It was "
            "designed to not be fully backward compatible with Python 2. Python has an "
            "extensive standard library, often described as 'batteries included'."
        ),
    },
    {
        "source": "wikipedia_rag",
        "title": "Retrieval-Augmented Generation",
        "content": (
            "Retrieval-Augmented Generation (RAG) is a technique for enhancing the "
            "accuracy and reliability of generative AI models with facts fetched from "
            "external sources. It combines an information retrieval component with "
            "a text generation model. When given a query, the retrieval system fetches "
            "relevant documents from a knowledge base, which are then passed to the "
            "generative model as context. RAG was introduced by Lewis et al. in 2020 "
            "and has become the dominant approach for building knowledge-intensive "
            "NLP applications. It reduces hallucinations by grounding the model in "
            "factual retrieved content."
        ),
    },
    {
        "source": "wikipedia_llm",
        "title": "Large language model",
        "content": (
            "A large language model (LLM) is a type of computational model designed "
            "for natural language processing tasks. LLMs are artificial neural networks "
            "that contain hundreds of billions (or more) parameters, and are trained "
            "on vast amounts of text data, generally through self-supervised and "
            "semi-supervised learning. As of 2024, the most capable LLMs are "
            "general-purpose models that can be used for a variety of tasks. Notable "
            "LLMs include GPT-4 (OpenAI), Claude (Anthropic), Gemini (Google), "
            "and LLaMA (Meta). LLMs process and generate text using transformer "
            "architectures trained on web-scale text corpora."
        ),
    },
]

## SQuAD 2.0 questions about the contexts

In [ ]:
SQUAD_QUESTIONS = [
    {
        "id": "SQ1",
        "question": "When was the Transformer architecture introduced?",
        "expected_source": "wikipedia_transformers",
        "expected_answer_contains": "2017",
    },
    {
        "id": "SQ2",
        "question": "What makes BERT different from GPT in terms of text processing?",
        "expected_source": "wikipedia_bert",
        "expected_answer_contains": "bidirectional",
    },
    {
        "id": "SQ3",
        "question": "Who created the Python programming language?",
        "expected_source": "wikipedia_python",
        "expected_answer_contains": "Guido",
    },
    {
        "id": "SQ4",
        "question": "What is RAG used for and what problem does it solve?",
        "expected_source": "wikipedia_rag",
        "expected_answer_contains": "hallucinations",
    },
    {
        "id": "SQ5",
        "question": "What is the difference between RAG and traditional language models?",
        "expected_source": "wikipedia_rag",
        "expected_answer_contains": None,  # cross-document question
    },
]


async def setup_vector_store(collection_name: str = "squad_crag") -> ChromaVectorStore:
    """Initialize and populate the vector store with SQuAD documents.

    Args:
        collection_name: ChromaDB collection name.

    Returns:
        ChromaVectorStore ready for search.
    """
    store = ChromaVectorStore(collection_name=collection_name)

    # Create LangChain documents with metadata
    docs = [
        Document(
            page_content=ctx["content"],
            metadata={
                "source": ctx["source"],
                "title": ctx["title"],
                "chunk_id": str(i),
                "dataset": "squad_2.0",
            },
        )
        for i, ctx in enumerate(SQUAD_CONTEXTS)
    ]

    # Index in ChromaDB
    print(f"  Indexing {len(docs)} documents in ChromaDB (collection: {collection_name})...")
    store.add_documents(docs)
    print(f"  ✓ Vectors indexed in: {collection_name}")

    return store


async def run_crag_query(
    pipeline: CRAGPipeline,
    question: dict,
) -> CRAGResult:
    """Run a CRAG query and display detailed results.

    Args:
        pipeline: Initialized CRAG pipeline.
        question: Question with expected metadata.

    Returns:
        CRAGResult with answer and sources.
    """
    return await pipeline.run(question["question"])


def print_crag_result(question: dict, result: CRAGResult) -> None:
    """Print CRAG results in a structured format."""
    print(f"\n[{question['id']}] {question['question']}")

    print("\n  Retrieved and graded sources:")
    for chunk in result.sources:
        score_bar = "█" * int(chunk.relevance_score * 10) + "░" * (
            10 - int(chunk.relevance_score * 10)
        )
        print(
            f"    [{score_bar}] {chunk.relevance_score:.2f} — "
            f"{chunk.source} (chunk {chunk.chunk_id})"
        )

    if result.used_web_fallback:
        print("  ⚠ Web fallback triggered (no chunk passed the 0.5 threshold)")
    else:
        relevant_count = len([c for c in result.sources if c.relevance_score >= 0.5])
        print(f"  ✓ {relevant_count} relevant chunks (score >= 0.5)")

    print("\n  CRAG answer:")
    print(f"  {result.answer[:400]}")

    # Check whether the expected source was used
    expected_src = question.get("expected_source")
    if expected_src:
        used_sources = [c.source for c in result.sources]
        found = expected_src in used_sources
        print(f"\n  Expected source ({expected_src}): {'✓ used' if found else '✗ not used'}")

    print("─" * 70)


async def main() -> None:
    print("=" * 70)
    print("  CRAG (Corrective RAG) — Dataset: SQuAD 2.0 (Wikipedia QA)")
    print("=" * 70)

    # Initialize vector store
    print("\n[Step 0: Vector Store Setup]")
    store = await setup_vector_store("squad_crag_example")

    # Create CRAG pipeline
    pipeline = CRAGPipeline(vector_store=store)
    print("  ✓ CRAG pipeline initialized")
    print("  Relevance threshold: 0.5 (chunks with score < 0.5 are discarded)")

    # Run queries
    print(f"\n[Queries over {len(SQUAD_QUESTIONS)} SQuAD questions]")

    for question in SQUAD_QUESTIONS:
        result = await run_crag_query(pipeline, question)
        print_crag_result(question, result)

    # Demonstrate web fallback (question without relevant documents)
    print("\n[Web Fallback Demonstration]")
    print("  Question without relevant documents in the knowledge base...")

    off_topic_result = await pipeline.run(
        "What are the best Michelin-starred sushi restaurants in Tokyo?"
    )
    print(f"  Fallback triggered: {off_topic_result.used_web_fallback}")
    print(
        f"  Fallback source: {off_topic_result.sources[0].source if off_topic_result.sources else 'N/A'}"
    )

    # CRAG pipeline summary
    print("\n[CRAG Flow — 5 steps]")
    steps = [
        ("1. RETRIEVE", "ChromaDB similarity_search(query, k=5)"),
        ("2. GRADE   ", "LLM scores each chunk 0.0-1.0"),
        ("3. FILTER  ", "Keep chunks with score >= 0.5"),
        ("4. DECIDE  ", "If all filtered out → web fallback"),
        ("5. GENERATE", "LLM answers with context + citations"),
    ]
    for step, desc in steps:
        print(f"  {step}: {desc}")


if __name__ == "__main__":
    asyncio.run(main())

---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()